# ch05 Bonus 09：Gradio 交互界面

> 对照官方 `ch05/06_user_interface`

## 一句话

用 **Gradio** 给训练好的模型套一个网页 UI，让非技术用户也能通过浏览器输入提示、调参、看生成结果。

## 原理

Gradio 把一个 Python 函数包装成 web 接口：定义输入组件（文本框、滑块）、输出组件（文本框），`gr.Interface(fn, inputs, outputs)` 一行起服务。LLM 场景下，输入是 prompt + 生成参数，输出是生成的文本。

In [ ]:
# 定义生成函数（这是 Gradio 要包装的核心）
import torch
import tiktoken
from src.gpt import GPTModel, GPT_CONFIG_124M

# 小配置 demo（真实场景用加载了 OpenAI 权重的 124M）
cfg = dict(GPT_CONFIG_124M)
cfg.update({"emb_dim": 128, "n_layers": 2, "n_heads": 4, "context_length": 128})
torch.manual_seed(123)
model = GPTModel(cfg)  # 未训练，仅演示 UI 流程
tok = tiktoken.get_encoding("gpt2")

def generate_text(prompt, max_tokens, temperature):
    """Gradio 包装的生成函数：输入 prompt + 参数，返回生成文本。"""
    model.eval()
    idx = torch.tensor([tok.encode(prompt)])
    with torch.no_grad():
        for _ in range(max_tokens):
            idx_cond = idx[:, -cfg["context_length"]:]
            logits = model(idx_cond)[:, -1, :]
            if temperature > 0:
                probs = torch.softmax(logits / max(temperature, 1e-5), dim=-1)
                next_id = torch.multinomial(probs, num_samples=1)
            else:
                next_id = torch.argmax(logits, dim=-1, keepdim=True)
            idx = torch.cat([idx, next_id], dim=1)
    return tok.decode(idx[0].tolist())

# 测试函数本身能工作
result = generate_text("Hello", max_tokens=10, temperature=0.8)
print("生成函数测试:", repr(result[:80]))

In [ ]:
# 构造 Gradio 界面（不实际启动服务，仅展示定义）
import gradio as gr

demo = gr.Interface(
    fn=generate_text,
    inputs=[
        gr.Textbox(label="提示词", value="I had a little"),
        gr.Slider(10, 200, value=50, step=10, label="最大生成长度"),
        gr.Slider(0.0, 1.5, value=0.8, step=0.1, label="温度（0=贪婪）"),
    ],
    outputs=gr.Textbox(label="生成结果", lines=8),
    title="GPT 文本生成",
    description="演示用的未训练小模型，加载 OpenAI 权重后效果更好。",
)

print("Gradio 界面已定义。")
print("启动方式（在终端运行）: demo.launch()  → 浏览器打开 http://127.0.0.1:7860")
print("\n💡 这里不实际 launch（会阻塞 notebook）；用户可取消注释下行启动。")
# demo.launch()  # 取消注释以启动服务